# 01 - Investigar, crear guion y generar voz

Este notebook busca páginas públicas por tema con ScrapeGraphAI/DuckDuckGo (sin claves de redes), analiza una URL opcional y genera narración con OmniVoice. Guarda estos secretos en Colab: `API_BASE_URL` (HTTPS publico), `WORKER_API_TOKEN`, `SCRIPT_SPACE_URL` (Space privado de Qwen), `VOICE_STUDIO_URL`, `HF_TOKEN` (solo lectura de ambos Spaces), `S3_ENDPOINT_URL`, `S3_BUCKET`, `S3_ACCESS_KEY_ID` y `S3_SECRET_ACCESS_KEY`.

El resultado sube el audio a R2 privado y crea automaticamente el trabajo de renderizado. El dashboard muestra el progreso de cada etapa.

In [ ]:
%pip install -q beautifulsoup4 boto3 requests gradio_client "git+https://github.com/javiertarazon/Scrapegraph-ai.git@c75c8084fae2d4f5ba01a8c218bc1168b67e3569"

In [ ]:
import os
from google.colab import userdata

REQUIRED = (
    "API_BASE_URL", "WORKER_API_TOKEN", "SCRIPT_SPACE_URL", "VOICE_STUDIO_URL",
    "HF_TOKEN", "S3_ENDPOINT_URL", "S3_BUCKET",
    "S3_ACCESS_KEY_ID", "S3_SECRET_ACCESS_KEY",
)
for name in REQUIRED:
    value = userdata.get(name)
    if not value:
        raise RuntimeError(f"Configura el secreto {name} en Colab")
    os.environ[name] = value
for name in ("API_BASE_URL", "SCRIPT_SPACE_URL", "VOICE_STUDIO_URL"):
    os.environ[name] = os.environ[name].rstrip("/")
if not os.environ["API_BASE_URL"].startswith("https://"):
    raise RuntimeError("API_BASE_URL debe ser HTTPS y accesible desde Colab")
API_HEADERS = {"Authorization": "Bearer " + os.environ["WORKER_API_TOKEN"]}

In [ ]:
from pathlib import Path
from urllib.parse import urljoin, urlparse
import ipaddress
import socket
import boto3
import requests
import threading
import tempfile
import time
from bs4 import BeautifulSoup
from gradio_client import Client
from ddgs import DDGS
from ddgs.exceptions import DDGSException
from scrapegraphai.utils.research_web import ResearchWebError, search_on_web


PLATFORM_SEARCH = {
    "youtube": {"label": "YouTube", "domains": ("youtube.com",)},
    "reddit": {"label": "Reddit", "domains": ("reddit.com",)},
    "x": {"label": "X", "domains": ("x.com", "twitter.com"), "query": "(site:x.com OR site:twitter.com)"},
    "instagram": {"label": "Instagram", "domains": ("instagram.com",)},
    "tiktok": {"label": "TikTok", "domains": ("tiktok.com",)},
    "facebook": {"label": "Facebook", "domains": ("facebook.com",)},
}
MAX_RESULTS_PER_PLATFORM = 2
MAX_SOCIAL_TEXT_CHARS = 1200


def validate_public_url(url):
    parsed = urlparse(url)
    if parsed.scheme not in ("http", "https") or not parsed.hostname:
        raise ValueError("source_url debe ser una URL HTTP/HTTPS valida")
    addresses = socket.getaddrinfo(parsed.hostname, parsed.port or (443 if parsed.scheme == "https" else 80))
    if not addresses or any(not ipaddress.ip_address(item[4][0].split("%")[0]).is_global for item in addresses):
        raise ValueError("source_url debe apuntar a un servidor publico")
    return parsed


def extract_article(source_url):
    current_url = source_url
    for _ in range(6):
        validate_public_url(current_url)
        response = requests.get(
            current_url,
            headers={"User-Agent": "MarketingContentOS/1.0 (article research)"},
            timeout=(10, 30),
            allow_redirects=False,
        )
        if response.is_redirect:
            location = response.headers.get("Location")
            if not location:
                raise RuntimeError("La fuente devolvio una redireccion sin destino")
            current_url = urljoin(current_url, location)
            continue
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        for element in soup(("script", "style", "noscript", "svg", "nav", "footer")):
            element.decompose()
        title_tag = soup.find("meta", attrs={"property": "og:title"})
        description_tag = soup.find("meta", attrs={"property": "og:description"}) or soup.find("meta", attrs={"name": "description"})
        title = (title_tag.get("content", "").strip() if title_tag else "") or (soup.title.get_text(" ", strip=True) if soup.title else "")
        text = soup.get_text(" ", strip=True)
        description = description_tag.get("content", "").strip() if description_tag else ""
        if description and description not in text:
            text = f"{text} {description}".strip()
        if len(text) < 100:
            raise RuntimeError("No se pudo extraer suficiente texto de la fuente")
        return {"url": current_url, "title": title, "text": text[:18000]}
    raise RuntimeError("La fuente excedio el maximo de redirecciones")


def matches_platform(url, platform):
    hostname = (urlparse(url).hostname or "").lower().rstrip(".")
    allowed_hosts = {
        host
        for domain in PLATFORM_SEARCH[platform]["domains"]
        for host in (domain, f"www.{domain}", f"m.{domain}")
    }
    return hostname in allowed_hosts


def search_public_platform(topic, platform):
    if platform not in PLATFORM_SEARCH:
        raise ValueError(f"Plataforma de investigación no admitida: {platform}")
    search_config = PLATFORM_SEARCH[platform]
    search_query = f"{search_config.get('query', 'site:' + search_config['domains'][0])} {topic}"
    search_error = ""
    try:
        urls = search_on_web(
            search_query,
            search_engine="duckduckgo",
            max_results=MAX_RESULTS_PER_PLATFORM,
            language="es",
        )
    except (ResearchWebError, TimeoutError, ImportError, ValueError) as error:
        urls = []
        search_error = str(error)[:240]

    sources = []
    errors = []
    for url in dict.fromkeys(urls):
        if not matches_platform(url, platform):
            continue
        try:
            article = extract_article(url)
        except (requests.RequestException, RuntimeError, ValueError, OSError) as error:
            errors.append(str(error)[:120])
            continue
        article["platform"] = platform
        article["source_type"] = "page"
        article["text"] = article["text"][:MAX_SOCIAL_TEXT_CHARS]
        sources.append(article)
        if len(sources) >= MAX_RESULTS_PER_PLATFORM:
            break

    if not sources:
        try:
            snippets = DDGS().text(search_query, max_results=MAX_RESULTS_PER_PLATFORM)
        except (DDGSException, TimeoutError, OSError) as error:
            errors.append(f"DuckDuckGo snippets: {str(error)[:120]}")
            snippets = []
        for snippet in snippets:
            url = snippet.get("href", "")
            text = snippet.get("body", "").strip()
            if not url or not matches_platform(url, platform) or len(text) < 40:
                continue
            sources.append({
                "platform": platform,
                "source_type": "search_snippet",
                "url": url,
                "title": snippet.get("title", "").strip(),
                "text": text[:MAX_SOCIAL_TEXT_CHARS],
            })
            if len(sources) >= MAX_RESULTS_PER_PLATFORM:
                break

    if sources and any(item["source_type"] == "page" for item in sources):
        status = "ok"
        message = ""
    elif sources:
        status = "snippet_only"
        message = "Se usan solo fragmentos del índice de DuckDuckGo; no se pudo leer el post completo."
    elif urls:
        status = "no_public_content"
        message = "; ".join(errors[:2]) or "Los resultados no eran páginas públicas de esta plataforma."
    elif search_error:
        status = "search_error"
        message = search_error
    else:
        status = "no_results"
        message = "DuckDuckGo no devolvió resultados para la búsqueda pública."
    return sources, {
        "platform": platform,
        "status": status,
        "result_count": len(sources),
        "message": message,
    }


def start_lease_heartbeat(job_id, lease_headers):
    stop = threading.Event()

    def renew_lease():
        while not stop.wait(60):
            try:
                response = requests.post(
                    f"{os.environ['API_BASE_URL']}/v1/jobs/{job_id}/heartbeat",
                    headers=lease_headers,
                    timeout=30,
                )
                response.raise_for_status()
            except requests.RequestException as error:
                print(f"No se pudo renovar la reserva del trabajo {job_id}: {error}")
                return

    thread = threading.Thread(target=renew_lease, daemon=True)
    thread.start()
    return stop, thread


def report_progress(job_id, lease_headers, stage, message, percent):
    response = requests.post(
        f"{os.environ['API_BASE_URL']}/v1/jobs/{job_id}/progress",
        headers=lease_headers,
        json={"stage": stage, "message": message, "percent": percent},
        timeout=30,
    )
    response.raise_for_status()


def process_one_research_job():
    response = requests.post(
        f"{os.environ['API_BASE_URL']}/v1/jobs/next",
        params={"worker": "research"}, headers=API_HEADERS, timeout=30,
    )
    response.raise_for_status()
    job = response.json()
    if job is None:
        print("No hay investigaciones pendientes.")
        return False

    job_id = job["job_id"]
    lease_headers = {**API_HEADERS, "X-Worker-Lease": job["lease_token"]}
    heartbeat_stop, heartbeat_thread = start_lease_heartbeat(job_id, lease_headers)
    reference_path = None
    try:
        request = job["request"]
        source_url = request.get("source_url")
        research_sources = []
        platform_status = []
        if source_url:
            report_progress(job_id, lease_headers, "research_source", "Extrayendo la URL de referencia", 10)
            article = extract_article(source_url)
            article["platform"] = "web"
            article["text"] = article["text"][:4000]
            research_sources.append(article)
            platform_status.append({"platform": "web", "status": "ok", "result_count": 1, "message": ""})
        selected_platforms = request.get("platforms") or (list(PLATFORM_SEARCH) if not source_url else [])
        for index, platform in enumerate(selected_platforms):
            label = PLATFORM_SEARCH.get(platform, {}).get("label", platform)
            progress = 10 + int(20 * index / max(1, len(selected_platforms)))
            report_progress(
                job_id, lease_headers, "research_search",
                f"Buscando contenido público en {label}", progress,
            )
            found_sources, status = search_public_platform(request["topic"], platform)
            research_sources.extend(found_sources)
            platform_status.append(status)
        if not research_sources:
            detail = "; ".join(
                f"{item['platform']}: {item['status']} - {item['message']}"
                for item in platform_status
            )
            raise RuntimeError(
                "No se pudo extraer contenido público. "
                "Las redes pueden no estar indexadas o bloquear el acceso anónimo. " + detail
            )
        report_progress(job_id, lease_headers, "research_script", "Analizando fuentes y generando guion", 32)
        source_context = "\n\n".join(
            f"Plataforma: {item['platform']}\nTipo de fuente: {item.get('source_type', 'page')}\nTitulo: {item['title']}\n"
            f"URL: {item['url']}\nContenido: {item['text']}"
            for item in research_sources
        )
        prompt = (
            "Eres un estratega de contenido y creador de videos virales. Analiza las fuentes publicas recopiladas sobre el tema.\n"
            "Identifica patrones de alto impacto: gancho inicial (hook), palabras clave que retienen la atencion y la idea central del video.\n"
            "Con base en esa investigacion, formula una idea unica y redacta un guion magnetico para video corto (Reels / TikTok / Shorts).\n"
            "Devuelve exactamente dos secciones en espanol con estos encabezados, sin formato Markdown ni texto adicional:\n"
            "INVESTIGACION:\n(Gancho detectado, palabras clave, idea central e investigacion)\n"
            "GUION:\n(Solo la narracion hablada, empezando con un hook demoledor en los primeros 3 segundos, directa y dinamica; no excedas aproximadamente "
            f"{request['duration_seconds']} segundos)\n\n"
            f"Tema: {request['topic']}\nEstilo: {request['style']}\n"
            f"Fuentes públicas recopiladas:\n{source_context}"
        )
        script_client = Client(
            os.environ["SCRIPT_SPACE_URL"], token=os.environ["HF_TOKEN"], verbose=False,
        )
        generated = script_client.predict(prompt, api_name="/generate")
        if not isinstance(generated, str):
            raise RuntimeError("El Space de guiones devolvio un formato inesperado")
        research_marker = "INVESTIGACION:\n"
        script_marker = "GUION:\n"
        if not generated.startswith(research_marker) or script_marker not in generated:
            raise RuntimeError("El Space no devolvio las secciones de investigacion y guion esperadas")
        research_text, script = generated[len(research_marker):].split(script_marker, 1)
        research_text, script = research_text.strip(), script.strip()
        if not research_text or not script:
            raise RuntimeError("El Space devolvio investigacion o guion vacios")

        report_progress(job_id, lease_headers, "voice_synthesis", "Preparando narración de voz", 55)
        storage = boto3.client(
            "s3", endpoint_url=os.environ["S3_ENDPOINT_URL"], region_name="auto",
            aws_access_key_id=os.environ["S3_ACCESS_KEY_ID"],
            aws_secret_access_key=os.environ["S3_SECRET_ACCESS_KEY"],
        )
        voice_reference = request.get("voice_reference")
        reference_transcript = ""
        if voice_reference:
            reference_path = Path(tempfile.gettempdir()) / f"{job_id}-voice-reference.wav"
            storage.download_file(os.environ["S3_BUCKET"], voice_reference["object_key"], str(reference_path))
            reference_transcript = voice_reference["transcript"]
        voice_client = Client(
            os.environ["VOICE_STUDIO_URL"],
            token=os.environ["HF_TOKEN"],
            verbose=False,
        )
        voice_path = voice_client.predict(
            script,
            "female",
            "young adult",
            "moderate pitch",
            str(reference_path) if reference_path else None,
            reference_transcript,
            api_name="/predict",
        )
        if not isinstance(voice_path, str) or not Path(voice_path).is_file():
            raise RuntimeError("El Space privado no devolvio el archivo de audio")
        voice_bytes = Path(voice_path).read_bytes()
        if not voice_bytes.startswith(b"RIFF"):
            raise RuntimeError("VoiceStudio devolvio un archivo que no es WAV")

        report_progress(job_id, lease_headers, "audio_storage", "Guardando narración en R2 privado", 85)
        audio_key = f"audio/{job_id}.wav"
        storage.put_object(Bucket=os.environ["S3_BUCKET"], Key=audio_key,
                           Body=voice_bytes, ContentType="audio/wav")
        complete = requests.post(
            f"{os.environ['API_BASE_URL']}/v1/jobs/{job_id}/complete",
            headers=lease_headers,
            json={"output": {"research": {"source": research_sources[0]["url"], "notes": research_text, "sources": [{"platform": item["platform"], "source_type": item.get("source_type", "page"), "title": item["title"], "url": item["url"]} for item in research_sources], "platform_status": platform_status}, "script": script, "audio_object_key": audio_key}},
            timeout=60,
        )
        complete.raise_for_status()
        print(f"Investigacion y voz listas; se encolo el render: {job_id}")
    except Exception as error:
        failed = requests.post(
            f"{os.environ['API_BASE_URL']}/v1/jobs/{job_id}/fail",
            headers=lease_headers, json={"error": str(error)[:1900]}, timeout=60,
        )
        failed.raise_for_status()
        raise RuntimeError(f"Trabajo {job_id} marcado como fallido: {error}") from error
    finally:
        heartbeat_stop.set()
        heartbeat_thread.join(timeout=5)
        if reference_path and reference_path.exists():
            reference_path.unlink()
    return True


POLL_INTERVAL_SECONDS = 30
print("Worker activo. Deten la celda para parar de consultar la cola.")
while True:
    try:
        processed = process_one_research_job()
    except Exception as error:
        print(f"Error del worker; volvera a intentarlo en {POLL_INTERVAL_SECONDS} s: {error}")
        time.sleep(POLL_INTERVAL_SECONDS)
        continue
    if not processed:
        time.sleep(POLL_INTERVAL_SECONDS)